## **Applio**
A simple, high-quality voice conversion tool focused on ease of use and performance.

[Support](https://discord.gg/urxFjYmYYh) — [GitHub](https://github.com/IAHispano/Applio) — [Terms of Use](https://github.com/IAHispano/Applio/blob/main/TERMS_OF_USE.md)

<br>

---

<br>

#### **Acknowledgments**

To all external collaborators for their special help in the following areas:
* Hina (Encryption method)
* Poopmaster (Extra section)
* Shirou (UV installer)
* Bruno5430 (AutoBackup code and general notebook maintenance)

#### **Disclaimer**
By using Applio, you agree to comply with ethical and legal standards, respect intellectual property and privacy rights, avoid harmful or prohibited uses, and accept full responsibility for any outcomes, while Applio disclaims liability and reserves the right to amend these terms.

### **Install Applio**
If the runtime restarts, re-run the installation steps.

In [ ]:
# @title Mount Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Setup runtime environment
from IPython.display import clear_output
import codecs

encoded_url = "uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/"
decoded_url = codecs.decode(encoded_url, "rot_13")

repo_name_encoded = "Nccyvb"
repo_name = codecs.decode(repo_name_encoded, "rot_13")

LOGS_PATH = f"/content/{repo_name}/logs"
BACKUPS_PATH = f"/content/drive/MyDrive/{repo_name}Backup"

%cd /content
!git config --global advice.detachedHead false
!git clone {decoded_url} --branch 3.6.2 --single-branch
%cd {repo_name}
clear_output()

!apt update -y
!apt install -y portaudio19-dev

print("Installing requirements...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel &> /dev/null

!python core.py "prerequisites" --models "True" --pretraineds_hifigan "True"
print("✅ Finished installing requirements!")


### **Start Applio**

In [ ]:
# @title Sync with Google Drive
# @markdown 💾 Run this cell to automatically Save/Load models from your mounted drive
# @title
# @markdown This will merge and link your `ApplioBackup` folder from gdrive to this notebook
from IPython.display import display, clear_output
from pathlib import Path

non_bak_folders = ["mute", "reference", "zips", "mute_spin", "mute_spin-v2"]
non_bak_path = "/tmp/rvc_logs"


def press_button(button):
  button.disabled = True


def get_date(path: Path):
  from datetime import datetime
  return datetime.fromtimestamp(int(path.stat().st_mtime))


def get_size(path: Path):
  !du -shx --apparent-size "{path}" > /tmp/size.txt
  return open("/tmp/size.txt").readlines().pop(0).split("	")[0] + "B"


def sync_folders(folder: Path, backup: Path):
  from ipywidgets import widgets
  from jupyter_ui_poll import ui_events
  from time import sleep

  local = widgets.VBox([
      widgets.Label(f"Local: {LOGS_PATH.removeprefix('/content/')}/{folder.name}/"),
      widgets.Label(f"Size: {get_size(folder)}"),
      widgets.Label(f"Last modified: {get_date(folder)}")
  ])
  remote = widgets.VBox([
      widgets.Label(f"Remote: {BACKUPS_PATH.removeprefix('/content/')}/{backup.name}/"),
      widgets.Label(f"Size: {get_size(backup)}"),
      widgets.Label(f"Last modified: {get_date(backup)}")
  ])
  separator = widgets.VBox([
      widgets.Label("|||"),
      widgets.Label("|||"),
      widgets.Label("|||")
  ])
  radio = widgets.RadioButtons(
      options=[
          "Save local model to drive",
          "Keep remote model"
      ]
  )
  button = widgets.Button(
      description="Sync",
      icon="upload",
      tooltip="Sync model"
  )
  button.on_click(press_button)

  clear_output()
  print(f"Your local model '{folder.name}' is in conflict with it's copy in Google Drive.")
  print("Please select which one you want to keep:")
  display(widgets.Box([local, separator, remote]))
  display(radio)
  display(button)

  with ui_events() as poll:
    while not button.disabled:
      poll(10)
      sleep(0.1)

  match radio.value:
    case "Save local model to drive":
      !rm -r "{backup}"
      !mv "{folder}" "{backup}"
    case "Keep remote model":
      !rm -r "{folder}"


if Path("/content/drive").is_mount():
  !mkdir -p "{BACKUPS_PATH}"
  !mkdir -p "{non_bak_path}"

  if not Path(LOGS_PATH).is_symlink():
    for folder in non_bak_folders:
      folder = Path(f"{LOGS_PATH}/{folder}")
      backup = Path(f"{BACKUPS_PATH}/{folder.name}")

      !mkdir -p "{folder}"
      !mv "{folder}" "{non_bak_path}" &> /dev/null
      !rm -rf "{folder}"
      folder = Path(f"{non_bak_path}/{folder.name}")
      if backup.exists() and backup.resolve() != folder.resolve():
        !rm -r "{backup}"
      !ln -s "{folder}" "{backup}" &> /dev/null

    for model in Path(LOGS_PATH).iterdir():
      if model.is_dir() and not model.is_symlink():
        backup = Path(f"{BACKUPS_PATH}/{model.name}")

        if model.name == ".ipynb_checkpoints":
          continue

        if backup.exists() and backup.is_dir():
          sync_folders(model, backup)
        else:
          !rm "{backup}"
          !mv "{model}" "{backup}"

    !rm -r "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"

    clear_output()
    print("✅ Models are synced!")

  else:
    !rm "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"
    clear_output()
    print("✅ Models already synced!")

else:
  print("❌ Drive is not mounted, skipping model syncing")
  print("To sync your models, first mount your Google Drive and re-run this cell")

In [ ]:
# @title **Web-Schnittstelle für VoiceAnime AI (Batch & Manuelle Eingabe)**
# @markdown Führe diese Zelle aus, um den Server zu starten.

import os
import shutil
import subprocess
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

# --- 0. PORT AUFRÄUMEN ---
# Beendet alte Server-Instanzen auf Port 8000
!fuser -k 8000/tcp > /dev/null 2>&1

# 1. Colab-Fix für Python 3.12
nest_asyncio.apply()

# 2. KONFIGURATION
ngrok_token = "Token eingeben" # @param {type:"string"}
APPLIO_PATH = "/content/Applio"
UPLOAD_DIR = "/content/uploads"
OUTPUT_DIR = "/content/outputs"
MODELS_DIR = "/content/models"

# Ordner erstellen, falls nicht vorhanden
for d in [UPLOAD_DIR, OUTPUT_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# 3. API INITIALISIERUNG
app = FastAPI()

# CORS erlauben, damit dein Browser vom PC aus auf Colab zugreifen darf
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/process-video/")
async def process_video(
    video: UploadFile = File(...),
    model_name: str = Form(...),
    pitch: int = Form(0)
):
    video_path = os.path.join(UPLOAD_DIR, video.filename)

    # Video lokal speichern
    with open(video_path, "wb") as buffer:
        shutil.copyfileobj(video.file, buffer)

    video_stem = os.path.splitext(video.filename)[0]
    temp_audio_path = os.path.join(UPLOAD_DIR, f"{video_stem}_temp.wav")
    isolated_folder = os.path.join(OUTPUT_DIR, "htdemucs", f"{video_stem}_temp")

    try:
        # SCHRITT 1: Audio aus Video extrahieren
        print(f"\n[1/4] Extrahiere Audio: {video.filename}")
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-vn", "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "2",
            temp_audio_path
        ], check=True, capture_output=True)

        # SCHRITT 2: Vocals isolieren (Demucs)
        print(f"[2/4] Trenne Stimmen (Demucs)...")
        subprocess.run([
            "demucs", "--two-stems=vocals", "-n", "htdemucs",
            temp_audio_path, "-o", OUTPUT_DIR
        ], check=True)

        vocal_path = os.path.join(isolated_folder, "vocals.wav")
        background_path = os.path.join(isolated_folder, "no_vocals.wav")
        ai_vocal_path = os.path.join(isolated_folder, "ai_vocals.wav")

        # SCHRITT 3: Applio Inferenz (Stimme ändern)
        print(f"[3/4] KI-Stimmwandlung: {model_name} (Pitch: {pitch})")

        # Pfad-Suche: Erst im neuen /models/, dann im alten /logs/
        model_pth = os.path.join(MODELS_DIR, model_name, f"{model_name}.pth")
        if not os.path.exists(model_pth):
            model_pth = os.path.join(APPLIO_PATH, "logs", model_name, f"{model_name}.pth")

        if not os.path.exists(model_pth):
            raise Exception(f"Modell '{model_name}' wurde nicht gefunden. Prüfe die Ordnerstruktur.")

        # Index-Datei suchen
        index_path = "None"
        model_dir = os.path.dirname(model_pth)
        for f in os.listdir(model_dir):
            if f.endswith(".index"):
                index_path = os.path.join(model_dir, f)
                break

        # Applio Core Aufruf
        subprocess.run([
            "python", "core.py", "infer",
            "--pth_path", model_pth,
            "--index_path", index_path,
            "--input_path", vocal_path,
            "--output_path", ai_vocal_path,
            "--pitch", str(pitch),
            "--f0_method", "rmvpe"
        ], cwd=APPLIO_PATH, check=True)

        # SCHRITT 4: Alles zusammenfügen
        print(f"[4/4] Merging Video...")
        final_output = os.path.join(OUTPUT_DIR, f"FINAL_{video.filename}")
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path, "-i", ai_vocal_path, "-i", background_path,
            "-filter_complex", "[1:a][2:a]amix=inputs=2:duration=first[aout]",
            "-map", "0:v:0", "-map", "[aout]", "-c:v", "copy", "-c:a", "aac",
            final_output
        ], check=True, capture_output=True)

        print(f"✅ FERTIG: {video.filename}")

        # --- CLEANUP (WICHTIG FÜR MEHRERE VIDEOS) ---
        # Löscht temporäre Audio-Dateien, lässt aber das fertige Video im Output
        if os.path.exists(isolated_folder):
            shutil.rmtree(isolated_folder)
        if os.path.exists(temp_audio_path):
            os.remove(temp_audio_path)
        if os.path.exists(video_path):
            os.remove(video_path)

        return FileResponse(final_output, media_type="video/mp4", filename=f"AI_Dub_{video.filename}")

    except Exception as e:
        print(f"❌ FEHLER BEI {video.filename}: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

# 4. SERVER START
if __name__ == "__main__":
    %cd /content/Applio
    ngrok.kill()
    ngrok.set_auth_token(ngrok_token)

    tunnel = ngrok.connect(8000)
    print("\n" + "="*60)
    print(f"1. KOPIERE DIESE URL IN DEINE index.html:\n{tunnel.public_url}")
    print("\n2. OEFFNE DIE URL EINMAL IM BROWSER & KLICKE 'VISIT SITE'")
    print("="*60 + "\n")

    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)

    await server.serve()